# PD-Disaggregation

1. LLM Inference 包含 Prefill 和 Decoding 两个计算任务， Prefill 是计算密集型任务，Decoding 是存储密集型任务
2. Chunked-Prefill 采用 Decoding 优先的推理策略，将 PD 计算任务融合在一个 batch 内进行 forward，这是常见的 PD 融合实例。
3. 另一种推理服务技术称之为 PD 分离，即 PD 节点各自运行其计算任务，以提高服务整体的吞吐率。PD 分离好处在于（1）不同的计算任务可以配置相应的物理硬件，例如 Prefill 节点上更快的计算核心的GPU，而 Decoding 节点则可以关注访存效率更高的通信系统。另外可以配置特定 GPU 数量 （2） PD 可以配置对应的 batch size。
4. PD 分离的缺陷在于，Prefill 后的 KV-Cache 需要传输到 Decoding 存在独特的系统开销
5. 当前的 PD 分离系统的 Prefill 节点实际上也可以采用 Chunked-Prefill 技术，不用过度争辩 PD 分离/融合 的优略，复杂的推理服务系统一定是融合和分离兼并的，系统设计是case-by-case 的
6. PD 分离之外，还有 Attention-FFN 分离（AF分离），训练分离（如 GRPO 训练/Agentic-RL 训练/ AReal 异步训练），推理服务的新的优化维度是 “分离”

关于 PD 分离的实现主要有两种

1. 基础版本：启动分离的 PD 进程，将 Prefill 的 KV-Cache 传输至 Decoding 节点。Decoding有等待的解码任务，则进行解码。
2. 分布式版本：PD 通常是 multi-node multi-gpu 环境部署的，考虑一种 KV-Cache 传输机制，Prefill 节点是生产者，将传输队列里传输 cache，而 Decoding 消费者则从传输队列里取 cache。在实现上采用异步传输的方式，使得通信-计算分离，提供系统任务重叠率。

实现代码

1. 非分布式版本
3. 在代码工程是实现一套基于 `Ray` 的 PD 分离推理系统。

当实现了分布式 PD 分离版本后，可以更深层次分析 PD 分离后的优化，如何管理分布式存储环境的 KV Cache？